# Impacts of Normalization Techniques

### Classification without RUS and SMOTE - (SVM, TSF, and GRU)

In [3]:
import pickle
import numpy as np

DATA_PATHS = {
    "per_file": {
        "train": "./final_split_data_FileNorm/train_set.pkl",
        "val":   "./final_split_data_FileNorm/val_set.pkl",
        "test":  "./final_split_data_FileNorm/test_set.pkl",
    },
    "per_row": {
        "train": "./final_split_data_RowNorm/train_set.pkl",
        "val":   "./final_split_data_RowNorm/val_set.pkl",
        "test":  "./final_split_data_RowNorm/test_set.pkl",
    },
    "hybrid": {
        "train": "./final_split_data_HybridNorm/train_set.pkl",
        "val":   "./final_split_data_HybridNorm/val_set.pkl",
        "test":  "./final_split_data_HybridNorm/test_set.pkl",
    },
}

def load_split(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data["X"].astype(np.float32), data["y"]

datasets = {}
for norm_name, splits in DATA_PATHS.items():
    X_train, y_train = load_split(splits["train"])
    X_val,   y_val   = load_split(splits["val"])
    X_test,  y_test  = load_split(splits["test"])
    datasets[norm_name] = {
        "X_train": X_train, "y_train": y_train,
        "X_val":   X_val,   "y_val":   y_val,
        "X_test":  X_test,  "y_test":  y_test,
    }

# ── Summary ────────────────────────────────────────────────────
print(f"{'Norm':<10} {'Train shape':<22} {'Val shape':<22} {'Test shape':<22} {'Class 0':>8} {'Class 1':>8} {'Imbalance':>10}")
print("─" * 110)
for norm_name, d in datasets.items():
    counts = np.bincount(d["y_train"])
    ratio  = counts[0] / counts[1]
    print(f"{norm_name:<10} {str(d['X_train'].shape):<22} {str(d['X_val'].shape):<22} {str(d['X_test'].shape):<22} "
          f"{counts[0]:>8} {counts[1]:>8} {ratio:>9.1f}x")

print(f"\n⚠️  Severe class imbalance detected (~{ratio:.0f}:1).")
print("   Consider using class_weight='balanced' in SVM,")
print("   and class_weight={{0:1, 1:{:.0f}}} in GRU.".format(ratio))

Norm       Train shape            Val shape              Test shape              Class 0  Class 1  Imbalance
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
per_file   (12455, 288, 10)       (1780, 288, 10)        (3559, 288, 10)           12337      118     104.6x
per_row    (12455, 288, 10)       (1780, 288, 10)        (3559, 288, 10)           12337      118     104.6x
hybrid     (12455, 288, 10)       (1780, 288, 10)        (3559, 288, 10)           12337      118     104.6x

⚠️  Severe class imbalance detected (~105:1).
   Consider using class_weight='balanced' in SVM,
   and class_weight={0:1, 1:105} in GRU.


### Vectorization for SVM

In [4]:
import os
import numpy as np
from joblib import Parallel, delayed
from sktime.transformations.panel.catch22 import Catch22

SAVE_DIR = "./catch22_features"
os.makedirs(SAVE_DIR, exist_ok=True)


def _catch22_chunk(X_chunk: np.ndarray) -> np.ndarray:
    transformer = Catch22(catch24=False)
    return transformer.fit_transform(X_chunk).to_numpy()


def extract_catch22(X: np.ndarray, label: str = "", n_jobs: int = -1, chunk_size: int = 500) -> np.ndarray:
    n_samples, n_timepoints, n_channels = X.shape

    if not np.isfinite(X).all():
        n_bad = (~np.isfinite(X)).sum()
        print(f"    ⚠️  {n_bad} non-finite values in {label} — replacing with per-channel median")
        X = X.copy()
        for c in range(n_channels):
            col = X[:, :, c]
            col[~np.isfinite(col)] = np.nanmedian(col)
            X[:, :, c] = col

    X_sktime = X.transpose(0, 2, 1).astype(np.float64)
    chunks   = [X_sktime[i:i + chunk_size] for i in range(0, n_samples, chunk_size)]

    print(f"    {label} — {n_samples} samples in {len(chunks)} chunks …", flush=True)

    results = Parallel(n_jobs=n_jobs)(
        delayed(_catch22_chunk)(chunk) for chunk in chunks
    )

    X_feat = np.vstack(results)

    expected_cols = 22 * n_channels
    assert X_feat.shape == (n_samples, expected_cols), (
        f"Unexpected catch22 output shape: {X_feat.shape}  "
        f"(expected ({n_samples}, {expected_cols}))"
    )
    return X_feat


# ── Sanity check ───────────────────────────────────────────────
print("Pre-extraction value range check:")
print(f"{'Norm':<10} {'Finite?':<10} {'Min':>15} {'Max':>15} {'NaN count':>12}")
print("─" * 65)
for norm_name, d in datasets.items():
    X = d["X_train"]
    print(f"{norm_name:<10} {str(np.isfinite(X).all()):<10} {X.min():>15.4f} {X.max():>15.4f} {(~np.isfinite(X)).sum():>12}")


# ── Extraction + save loop ─────────────────────────────────────
print("\nExtracting & saving catch22 features …")

datasets_svm = {}

for norm_name, d in datasets.items():

    print(f"\n[{norm_name}] extracting train …", flush=True)
    X_tr = extract_catch22(d["X_train"], label=f"{norm_name}/train", n_jobs=-1, chunk_size=200)

    print(f"[{norm_name}] extracting val …", flush=True)
    X_va = extract_catch22(d["X_val"],   label=f"{norm_name}/val",   n_jobs=-1, chunk_size=200)

    print(f"[{norm_name}] extracting test …", flush=True)
    X_te = extract_catch22(d["X_test"],  label=f"{norm_name}/test",  n_jobs=-1, chunk_size=200)

    np.savez(os.path.join(SAVE_DIR, f"{norm_name}_train.npz"), X=X_tr, y=d["y_train"])
    np.savez(os.path.join(SAVE_DIR, f"{norm_name}_val.npz"),   X=X_va, y=d["y_val"])
    np.savez(os.path.join(SAVE_DIR, f"{norm_name}_test.npz"),  X=X_te, y=d["y_test"])

    datasets_svm[norm_name] = {
        "X_train": X_tr, "y_train": d["y_train"],
        "X_val":   X_va, "y_val":   d["y_val"],
        "X_test":  X_te, "y_test":  d["y_test"],
    }

    print(f"[{norm_name}] ✓  train {X_tr.shape}  val {X_va.shape}  test {X_te.shape}  → {SAVE_DIR}/")

print(f"\n✅  Catch22 extraction complete.")
print(f"    Folder : {os.path.abspath(SAVE_DIR)}/")
print(f"    Files  : <norm>_train.npz, <norm>_val.npz, <norm>_test.npz  (keys: 'X', 'y')")
print(f"    Shape  : 22 features × 10 channels = 220 cols per sample")

Pre-extraction value range check:
Norm       Finite?                Min             Max    NaN count
─────────────────────────────────────────────────────────────────
per_file   True               -6.5219        248.2872            0
per_row    True              -16.9411         16.9411            0
hybrid     True               -6.5219        248.1381            0

Extracting & saving catch22 features …

[per_file] extracting train …
    per_file/train — 12455 samples in 63 chunks …
[per_file] extracting val …
    per_file/val — 1780 samples in 9 chunks …
[per_file] extracting test …
    per_file/test — 3559 samples in 18 chunks …
[per_file] ✓  train (12455, 220)  val (1780, 220)  test (3559, 220)  → ./catch22_features/

[per_row] extracting train …
    per_row/train — 12455 samples in 63 chunks …
[per_row] extracting val …
    per_row/val — 1780 samples in 9 chunks …
[per_row] extracting test …
    per_row/test — 3559 samples in 18 chunks …
[per_row] ✓  train (12455, 220)  val (1780,

### Missing value imputation from Catch22

In [1]:
from sklearn.impute import SimpleImputer
import os
import numpy as np

SAVE_DIR = "./catch22_features"
datasets_svm = {}

for fname in sorted(os.listdir(SAVE_DIR)):
    if not fname.endswith("_train.npz"):
        continue
    norm_name = fname.replace("_train.npz", "")

    paths = {
        "train": os.path.join(SAVE_DIR, f"{norm_name}_train.npz"),
        "val":   os.path.join(SAVE_DIR, f"{norm_name}_val.npz"),
        "test":  os.path.join(SAVE_DIR, f"{norm_name}_test.npz"),
    }
    if not all(os.path.exists(p) for p in paths.values()):
        print(f"  ⚠️  [{norm_name}] missing val or test file — skipping")
        continue

    X_tr, y_tr = np.load(paths["train"])["X"], np.load(paths["train"])["y"]
    X_va, y_va = np.load(paths["val"])["X"],   np.load(paths["val"])["y"]
    X_te, y_te = np.load(paths["test"])["X"],  np.load(paths["test"])["y"]

    n_bad_tr = (~np.isfinite(X_tr)).sum()
    n_bad_va = (~np.isfinite(X_va)).sum()
    n_bad_te = (~np.isfinite(X_te)).sum()

    if n_bad_tr > 0 or n_bad_va > 0 or n_bad_te > 0:
        print(f"  ⚠️  [{norm_name}] non-finite values (NaN/Inf) — "
              f"train: {n_bad_tr}  val: {n_bad_va}  test: {n_bad_te}  → imputing")

        # replace Inf with NaN so SimpleImputer can handle them
        X_tr = np.where(np.isfinite(X_tr), X_tr, np.nan)
        X_va = np.where(np.isfinite(X_va), X_va, np.nan)
        X_te = np.where(np.isfinite(X_te), X_te, np.nan)

        imp  = SimpleImputer(strategy="median")
        X_tr = imp.fit_transform(X_tr)   # fit on train only
        X_va = imp.transform(X_va)       # apply train median to val
        X_te = imp.transform(X_te)       # apply train median to test

        np.savez(paths["train"], X=X_tr, y=y_tr)
        np.savez(paths["val"],   X=X_va, y=y_va)
        np.savez(paths["test"],  X=X_te, y=y_te)
        print(f"  ✓  [{norm_name}] cleaned files saved")
    else:
        print(f"  ✓  [{norm_name}] no NaN or Inf — files unchanged")

    datasets_svm[norm_name] = {
        "X_train": X_tr, "y_train": y_tr,
        "X_val":   X_va, "y_val":   y_va,
        "X_test":  X_te, "y_test":  y_te,
    }

print(f"\n✅  datasets_svm loaded: {list(datasets_svm.keys())}")
print(f"    Train shape : {list(datasets_svm.values())[0]['X_train'].shape}")
print(f"    Val shape   : {list(datasets_svm.values())[0]['X_val'].shape}")
print(f"    Test shape  : {list(datasets_svm.values())[0]['X_test'].shape}")

  ✓  [hybrid] no NaN or Inf — files unchanged
  ✓  [per_file] no NaN or Inf — files unchanged
  ✓  [per_row] no NaN or Inf — files unchanged

✅  datasets_svm loaded: ['hybrid', 'per_file', 'per_row']
    Train shape : (12455, 220)
    Val shape   : (1780, 220)
    Test shape  : (3559, 220)


### Classification

In [2]:
# ══════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ══════════════════════════════════════════════════════════════
import numpy as np
from sklearn.metrics import confusion_matrix
from sklearn.svm import SVC
import time
import os

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)


def compute_metrics(y_true, y_pred):
    cm             = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm.ravel()

    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy  = (TP + TN) / (TP + TN + FP + FN)

    tss  = recall - FP / (FP + TN) if (FP + TN) > 0 else 0.0

    expected = ((TP + FN) * (TP + FP) + (TN + FP) * (TN + FN)) / (TP + TN + FP + FN) ** 2
    hss1     = (accuracy - expected) / (1 - expected) if (1 - expected) > 0 else 0.0

    denom = ((TP + FN) * (FN + TN) + (TP + FP) * (FP + TN))
    hss2  = 2 * (TP * TN - FP * FN) / denom if denom > 0 else 0.0

    hits_random = (TP + FP) * (TP + FN) / (TP + TN + FP + FN)
    gss = (TP - hits_random) / (TP + FP + FN - hits_random) if (TP + FP + FN - hits_random) > 0 else 0.0

    return {
        'TP': int(TP), 'TN': int(TN), 'FP': int(FP), 'FN': int(FN),
        'tss': tss, 'hss1': hss1, 'hss2': hss2, 'gss': gss,
        'recall': recall, 'f1': f1, 'accuracy': accuracy
    }


def train_and_evaluate(model, X_train, y_train, X_test, y_test):
    t0         = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0

    t0         = time.time()
    y_pred     = model.predict(X_test)
    infer_time = time.time() - t0

    metrics = compute_metrics(y_test, y_pred)
    metrics['train_time'] = train_time
    metrics['infer_time'] = infer_time
    return metrics


def save_results(metrics_list, filepath):
    with open(filepath, 'w') as f:
        for m in metrics_list:
            line = (f"{m['TP']},{m['TN']},{m['FP']},{m['FN']},"
                    f"{m['tss']:.6f},{m['hss1']:.6f},{m['hss2']:.6f},{m['gss']:.6f},"
                    f"{m['recall']:.6f},{m['f1']:.6f},{m['accuracy']:.6f},"
                    f"{m['train_time']:.4f},{m['infer_time']:.4f}")
            f.write(line + "\n")


def print_results(metrics_list, title):
    keys = ['tss', 'hss1', 'hss2', 'gss', 'recall', 'f1', 'accuracy', 'train_time', 'infer_time']
    print(f"\n{'─'*55}")
    print(f"  {title}")
    print(f"{'─'*55}")
    for i, m in enumerate(metrics_list):
        print(f"  Run {i+1}: TP={m['TP']}  TN={m['TN']}  FP={m['FP']}  FN={m['FN']}")
        print(f"         TSS={m['tss']:.4f}  HSS1={m['hss1']:.4f}  HSS2={m['hss2']:.4f}  GSS={m['gss']:.4f}")
        print(f"         Recall={m['recall']:.4f}  F1={m['f1']:.4f}  Acc={m['accuracy']:.4f}")
        print(f"         Train={m['train_time']:.2f}s  Infer={m['infer_time']:.4f}s")
        print()
    print(f"  ── Average of {len(metrics_list)} runs ──")
    for k in keys:
        avg = np.mean([m[k] for m in metrics_list])
        print(f"  {k:<12} : {avg:.4f}")
    print(f"{'─'*55}")

### SVM

In [5]:
# ══════════════════════════════════════════════════════════════
# CLASSIFIER & HYPERPARAMETER GRID
# ══════════════════════════════════════════════════════════════

DATASET_NAMES = {
    "per_file" : "per_feature_zscore",
    "per_row"  : "per_sample_zscore",
    "hybrid"   : "hybrid_method",
}

from sklearn.svm import SVC, LinearSVC

SVM_GRID = [
    {"model": LinearSVC, "params": {"C": 0.5, "class_weight": "balanced", "max_iter": 3000}},
    {"model": LinearSVC, "params": {"C": 1,   "class_weight": "balanced", "max_iter": 3000}},
    {"model": LinearSVC, "params": {"C": 2,   "class_weight": "balanced", "max_iter": 3000}},
    {"model": LinearSVC, "params": {"C": 3,   "class_weight": None,       "max_iter": 3000}},
    {"model": LinearSVC, "params": {"C": 5,   "class_weight": None,       "max_iter": 3000}},
    {"model": LinearSVC, "params": {"C": 0.1, "class_weight": None,       "max_iter": 3000}},
    {"model": SVC,       "params": {"kernel": "rbf", "C": 1,  "gamma": "scale", "class_weight": "balanced", "cache_size": 2000}},
    {"model": SVC,       "params": {"kernel": "rbf", "C": 10, "gamma": "scale", "class_weight": "balanced", "cache_size": 2000}},
]

def combo_label(combo):
    ModelClass = combo["model"]
    params     = combo["params"]
    cw         = "balanced" if params.get("class_weight") == "balanced" else "none"
    if ModelClass == LinearSVC:
        return f"LinearSVC  C={params['C']}  cw={cw}"
    else:
        return f"RBF        C={params['C']}  cw={cw}"

In [6]:
print(f"{'═'*60}")
print(f"  Classifier : SVM")
print(f"{'═'*60}")

# ── Step 1: Find best combo on hybrid val set only ────────────
print(f"\n  Searching {len(SVM_GRID)} combinations on hybrid val set …")

d_hybrid    = datasets_svm["hybrid"]
best_params      = None
best_model_class = None
best_tss         = -999

for combo in SVM_GRID:
    ModelClass = combo["model"]
    params     = combo["params"]
    model      = ModelClass(**params)
    t0         = time.time()
    model.fit(d_hybrid["X_train"], d_hybrid["y_train"])
    y_pred     = model.predict(d_hybrid["X_val"])
    elapsed    = time.time() - t0
    m          = compute_metrics(d_hybrid["y_val"], y_pred)
    print(f"    {combo_label(combo):<25}  →  TSS={m['tss']:.4f}  ({elapsed:.1f}s)")

    if m['tss'] > best_tss:
        best_tss         = m['tss']
        best_params      = params
        best_model_class = ModelClass

print(f"\n  ✅ Best model  : {best_model_class.__name__}")
print(f"     Best params : {combo_label({'model': best_model_class, 'params': best_params})}")
print(f"     Best TSS    : {best_tss:.4f}")

# ── Step 2: Run experiments on all three datasets ─────────────
print(f"\n  Running 2 experiments with best params on all datasets …")

for norm_key, norm_label in DATASET_NAMES.items():
    if norm_key not in datasets_svm:
        print(f"  [{norm_key}] not found — skipping")
        continue

    d = datasets_svm[norm_key]
    print(f"\n  ── Dataset: {norm_label} ──")

    metrics_list = []
    for run in range(2):
        model   = best_model_class(**best_params)
        metrics = train_and_evaluate(
            model,
            d["X_train"], d["y_train"],
            d["X_test"],  d["y_test"]
        )
        metrics_list.append(metrics)
        print(f"    Run {run+1} — TSS={metrics['tss']:.4f}  F1={metrics['f1']:.4f}  "
              f"Train={metrics['train_time']:.2f}s")

    filepath = os.path.join(RESULTS_DIR, f"svm_{norm_key}.txt")
    save_results(metrics_list, filepath)
    print_results(metrics_list, f"SVM | {norm_label}")

print(f"\n\n✅  All SVM experiments complete.")
print(f"    Results saved to : {os.path.abspath(RESULTS_DIR)}/")
print(f"    Files : {sorted([f for f in os.listdir(RESULTS_DIR) if f.endswith('.txt')])}")

════════════════════════════════════════════════════════════
  Classifier : SVM
════════════════════════════════════════════════════════════

  Searching 8 combinations on hybrid val set …
    LinearSVC  C=0.5  cw=balanced  →  TSS=0.4372  (121.8s)
    LinearSVC  C=1  cw=balanced  →  TSS=0.4983  (190.6s)
    LinearSVC  C=2  cw=balanced  →  TSS=0.4943  (160.8s)


/Users/samskanderi/anaconda3/lib/python3.11/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


    LinearSVC  C=3  cw=none    →  TSS=-0.0057  (32.7s)


/Users/samskanderi/anaconda3/lib/python3.11/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


    LinearSVC  C=5  cw=none    →  TSS=-0.0057  (38.2s)
    LinearSVC  C=0.1  cw=none  →  TSS=-0.0057  (23.8s)
    RBF        C=1  cw=balanced  →  TSS=0.0347  (9.7s)
    RBF        C=10  cw=balanced  →  TSS=-0.0715  (6.3s)

  ✅ Best model  : LinearSVC
     Best params : LinearSVC  C=1  cw=balanced
     Best TSS    : 0.4983

  Running 2 experiments with best params on all datasets …

  ── Dataset: per_feature_zscore ──


/Users/samskanderi/anaconda3/lib/python3.11/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


    Run 1 — TSS=0.2877  F1=0.0508  Train=211.23s


/Users/samskanderi/anaconda3/lib/python3.11/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


    Run 2 — TSS=0.2877  F1=0.0508  Train=209.68s

───────────────────────────────────────────────────────
  SVM | per_feature_zscore
───────────────────────────────────────────────────────
  Run 1: TP=15  TN=2984  FP=541  FN=19
         TSS=0.2877  HSS1=0.0334  HSS2=0.0334  GSS=0.0170
         Recall=0.4412  F1=0.0508  Acc=0.8427
         Train=211.23s  Infer=0.0017s

  Run 2: TP=15  TN=2984  FP=541  FN=19
         TSS=0.2877  HSS1=0.0334  HSS2=0.0334  GSS=0.0170
         Recall=0.4412  F1=0.0508  Acc=0.8427
         Train=209.68s  Infer=0.0106s

  ── Average of 2 runs ──
  tss          : 0.2877
  hss1         : 0.0334
  hss2         : 0.0334
  gss          : 0.0170
  recall       : 0.4412
  f1           : 0.0508
  accuracy     : 0.8427
  train_time   : 210.4537
  infer_time   : 0.0061
───────────────────────────────────────────────────────

  ── Dataset: per_sample_zscore ──
    Run 1 — TSS=0.3141  F1=0.0604  Train=68.56s
    Run 2 — TSS=0.3141  F1=0.0604  Train=68.90s

──────────────

### GRU 

In [14]:
# ══════════════════════════════════════════════════════════════
# HELPER FUNCTIONS (reuse from SVM section)
# compute_metrics, train_and_evaluate, save_results, print_results
# ══════════════════════════════════════════════════════════════

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers.legacy import Adam
import time
import os
import warnings
warnings.filterwarnings('ignore')

# ── GRU model builder ─────────────────────────────────────────
def build_gru(input_shape, units=64, dropout=0.3, layers=1,
              lr=0.001, class_weight_ratio=13):
    model = Sequential()

    for i in range(layers):
        return_seq = (i < layers - 1)
        if i == 0:
            model.add(GRU(units, return_sequences=return_seq,
                          input_shape=input_shape))
        else:
            model.add(GRU(units, return_sequences=return_seq))
        model.add(BatchNormalization())
        model.add(Dropout(dropout))

    model.add(Dense(32, activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model


# ── 8 hyperparameter combinations ─────────────────────────────
GRU_GRID = [
    {"units": 32,  "dropout": 0.3, "layers": 1, "lr": 0.001,  "batch_size": 64},
    {"units": 64,  "dropout": 0.3, "layers": 1, "lr": 0.001,  "batch_size": 64},
    {"units": 128, "dropout": 0.3, "layers": 1, "lr": 0.001,  "batch_size": 32},
    {"units": 64,  "dropout": 0.5, "layers": 1, "lr": 0.001,  "batch_size": 64},
    {"units": 64,  "dropout": 0.1, "layers": 2, "lr": 0.001,  "batch_size": 64},
    {"units": 128, "dropout": 0.1, "layers": 2, "lr": 0.001,  "batch_size": 32},
    {"units": 64,  "dropout": 0.3, "layers": 4, "lr": 0.001,  "batch_size": 32},
    {"units": 128, "dropout": 0.1, "layers": 4, "lr": 0.0001, "batch_size": 64},
]

def gru_label(p):
    return (f"units={p['units']}  layers={p['layers']}  "
            f"drop={p['dropout']}  lr={p['lr']}  bs={p['batch_size']}")


# ── GRU train and evaluate ────────────────────────────────────
def train_and_evaluate_gru(params, X_train, y_train, X_val,
                            y_val, X_test, y_test, class_ratio=13):
    input_shape = (X_train.shape[1], X_train.shape[2])
    cw          = {0: 1, 1: int(class_ratio)}

    model = build_gru(
        input_shape  = input_shape,
        units        = params["units"],
        dropout      = params["dropout"],
        layers       = params["layers"],
        lr           = params["lr"],
    )

    es = EarlyStopping(monitor='val_loss', patience=5,
                       restore_best_weights=True, verbose=0)

    t0 = time.time()
    model.fit(
        X_train, y_train,
        validation_data = (X_val, y_val),
        epochs          = 50,
        batch_size      = params["batch_size"],
        class_weight    = cw,
        callbacks       = [es],
        verbose         = 0,
    )
    train_time = time.time() - t0

    t0     = time.time()
    y_prob = model.predict(X_test, verbose=0).flatten()
    y_pred = (y_prob >= 0.5).astype(int)
    infer_time = time.time() - t0

    metrics = compute_metrics(y_test, y_pred)
    metrics['train_time'] = train_time
    metrics['infer_time'] = infer_time
    return metrics, model


def predict_gru(model, X_test, y_test):
    t0     = time.time()
    y_prob = model.predict(X_test, verbose=0).flatten()
    y_pred = (y_prob >= 0.5).astype(int)
    infer_time = time.time() - t0
    metrics = compute_metrics(y_test, y_pred)
    metrics['infer_time'] = infer_time
    return metrics

In [15]:
# ══════════════════════════════════════════════════════════════
# LOAD TIME SERIES DATASETS (3D)
# ══════════════════════════════════════════════════════════════
import pickle

TS_DATA_PATHS = {
    "per_file": {
        "train": "./final_split_data_FileNorm/train_set.pkl",
        "val":   "./final_split_data_FileNorm/val_set.pkl",
        "test":  "./final_split_data_FileNorm/test_set.pkl",
    },
    "per_row": {
        "train": "./final_split_data_RowNorm/train_set.pkl",
        "val":   "./final_split_data_RowNorm/val_set.pkl",
        "test":  "./final_split_data_RowNorm/test_set.pkl",
    },
    "hybrid": {
        "train": "./final_split_data_HybridNorm/train_set.pkl",
        "val":   "./final_split_data_HybridNorm/val_set.pkl",
        "test":  "./final_split_data_HybridNorm/test_set.pkl",
    },
}

def load_ts_split(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data["X"].astype(np.float32), data["y"].astype(np.float32)

datasets_gru = {}
for norm_name, splits in TS_DATA_PATHS.items():
    X_train, y_train = load_ts_split(splits["train"])
    X_val,   y_val   = load_ts_split(splits["val"])
    X_test,  y_test  = load_ts_split(splits["test"])
    datasets_gru[norm_name] = {
        "X_train": X_train, "y_train": y_train,
        "X_val":   X_val,   "y_val":   y_val,
        "X_test":  X_test,  "y_test":  y_test,
    }
    print(f"  {norm_name:<10}  train={X_train.shape}  val={X_val.shape}  test={X_test.shape}")

  per_file    train=(12455, 288, 10)  val=(1780, 288, 10)  test=(3559, 288, 10)
  per_row     train=(12455, 288, 10)  val=(1780, 288, 10)  test=(3559, 288, 10)
  hybrid      train=(12455, 288, 10)  val=(1780, 288, 10)  test=(3559, 288, 10)


In [16]:
# ══════════════════════════════════════════════════════════════
# RUN GRU EXPERIMENTS
# ══════════════════════════════════════════════════════════════

DATASET_NAMES = {
    "per_file" : "per_feature_zscore",
    "per_row"  : "per_sample_zscore",
    "hybrid"   : "hybrid_method",
}

print(f"{'═'*60}")
print(f"  Classifier : GRU")
print(f"{'═'*60}")

# ── Step 1: Val search on hybrid only ─────────────────────────
print(f"\n  Searching {len(GRU_GRID)} combinations on hybrid val set …\n")

d_hybrid     = datasets_gru["hybrid"]
best_params  = None
best_tss     = -999

# compute class ratio from training set
class_ratio  = int((d_hybrid["y_train"] == 0).sum() /
                    (d_hybrid["y_train"] == 1).sum())
print(f"  Class ratio (0:1) = {class_ratio}:1\n")

for i, params in enumerate(GRU_GRID):
    print(f"  [{i+1}/8] {gru_label(params)}")
    metrics, _ = train_and_evaluate_gru(
        params,
        d_hybrid["X_train"], d_hybrid["y_train"],
        d_hybrid["X_val"],   d_hybrid["y_val"],
        d_hybrid["X_val"],   d_hybrid["y_val"],   # evaluate on val
        class_ratio = class_ratio
    )
    print(f"         TSS={metrics['tss']:.4f}  F1={metrics['f1']:.4f}  "
          f"Train={metrics['train_time']:.1f}s\n")

    if metrics['tss'] > best_tss:
        best_tss    = metrics['tss']
        best_params = params

print(f"  ✅ Best params : {gru_label(best_params)}")
print(f"     Best TSS   : {best_tss:.4f}")

# ── Step 2: Run experiments on all three datasets ──────────────
print(f"\n  Running 2 experiments with best params on all datasets …")

for norm_key, norm_label in DATASET_NAMES.items():
    if norm_key not in datasets_gru:
        print(f"  [{norm_key}] not found — skipping")
        continue

    d = datasets_gru[norm_key]
    cr = int((d["y_train"] == 0).sum() / (d["y_train"] == 1).sum())

    print(f"\n  ── Dataset: {norm_label} ──")

    metrics_list = []

    for run in range(2):
        print(f"    Run {run+1} …", flush=True)
        metrics, _ = train_and_evaluate_gru(
            best_params,
            d["X_train"], d["y_train"],
            d["X_val"],   d["y_val"],
            d["X_test"],  d["y_test"],
            class_ratio = cr
        )
        metrics_list.append(metrics)
        print(f"    Run {run+1} — TSS={metrics['tss']:.4f}  F1={metrics['f1']:.4f}  "
              f"Train={metrics['train_time']:.1f}s")

    filepath = os.path.join(RESULTS_DIR, f"gru_{norm_key}.txt")
    save_results(metrics_list, filepath)
    print_results(metrics_list, f"GRU | {norm_label}")

print(f"\n\n✅  All GRU experiments complete.")
print(f"    Results saved to : {os.path.abspath(RESULTS_DIR)}/")
print(f"    Files : {sorted([f for f in os.listdir(RESULTS_DIR) if f.endswith('.txt')])}")

════════════════════════════════════════════════════════════
  Classifier : GRU
════════════════════════════════════════════════════════════

  Searching 8 combinations on hybrid val set …

  Class ratio (0:1) = 104:1

  [1/8] units=32  layers=1  drop=0.3  lr=0.001  bs=64
         TSS=0.3702  F1=0.0792  Train=158.4s

  [2/8] units=64  layers=1  drop=0.3  lr=0.001  bs=64
         TSS=0.4088  F1=0.1194  Train=126.7s

  [3/8] units=128  layers=1  drop=0.3  lr=0.001  bs=32
         TSS=0.5047  F1=0.0789  Train=446.7s

  [4/8] units=64  layers=1  drop=0.5  lr=0.001  bs=64
         TSS=0.3216  F1=0.0765  Train=78.3s

  [5/8] units=64  layers=2  drop=0.1  lr=0.001  bs=64
         TSS=0.3691  F1=0.0784  Train=242.7s

  [6/8] units=128  layers=2  drop=0.1  lr=0.001  bs=32
         TSS=0.4069  F1=0.0744  Train=669.8s

  [7/8] units=64  layers=4  drop=0.3  lr=0.001  bs=32
         TSS=0.3311  F1=0.0590  Train=970.4s

  [8/8] units=128  layers=4  drop=0.1  lr=0.0001  bs=64
         TSS=0.3475  F1=